# Chapter 21: Topology of Surfaces

Source orientation: printed pages 379-395; PDF pages 397-413. This notebook is an original standalone lesson built from the chapter's topic sequence. It uses the source for terminology and coverage only; it does not reproduce source prose, page crops, exercise text, or figures.

## Chapter Goal

The chapter goal is to turn closed surfaces and map coloring into inspectable invariants: Euler characteristic, orientability, regular-map incidence counts, low-degree faces for coloring induction, Heawood's sufficient color bound, and the complete-graph witnesses that force the full number of colors on many surfaces.

## Computational Translation Guide

- A surface is represented by a cell count `V - E + F`, a gluing pattern, or a frame transported around a loop.
- A handle reduces `chi` by 2; a cross-cap reduces `chi` by 1. Those two moves separate orientable and nonorientable classification data.
- A regular map of type `{p, q}` becomes the incidence equations `pF = 2E = qV` together with Euler's formula.
- Map coloring becomes graph coloring of the face-adjacency graph: adjacent faces must receive different colors.
- The six-color theorem is an induction scaffold: Euler's formula forces a face with at most five neighbors, so a sixth color can restore the removed face.
- For arbitrary surfaces, Heawood's number is a computable upper bound; complete graphs give sharpness witnesses except for the Klein-bottle exception.

## Visual Storyboard Implemented

| Section | Artifact | Inspection target | Invariant or check |
| --- | --- | --- | --- |
| Orientable and nonorientable surfaces | `surface-characteristic-dictionary.svg`, `orientability-frame-transport.html` | handles, cross-caps, and frame reversal | `chi=2-2g`, `chi=2-q`, transported frame dot products |
| Regular maps | `regular-map-counts-and-duals.svg` | how `{p,q}` fixes `V,E,F` away from `chi=0` | symbolic regular-map count derivation |
| Four-color problem | `four-color-face-adjacency-examples.svg` | face adjacency as graph coloring | exact chromatic numbers for small witness graphs |
| Six-color theorem | `six-color-induction-step.svg` | low-edge face removal and recoloring | average face-edge inequality from Euler characteristic |
| Arbitrary surfaces | `heawood-bound-by-characteristic.svg` | how the sufficient color count grows as `chi` falls | Heawood quadratic identity |
| Full color counts | `full-color-witness-cliques.svg` | complete-graph witnesses and the Klein-bottle exception | genus formulas and exact clique chromatic numbers |


In [1]:
from __future__ import annotations

from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sympy as sp
from IPython.display import Markdown, display

CHAPTER_NO = 21
HERE = Path.cwd().resolve()
root_candidates = [HERE, HERE / "Introduction-to-Geometry"]
for parent in HERE.parents:
    root_candidates.extend([parent, parent / "Introduction-to-Geometry"])
BOOK_ROOT = None
for candidate in root_candidates:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate.resolve()
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate Introduction-to-Geometry book root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (
    assert_artifacts,
    book_relative,
    display_artifact,
    ensure_chapter_artifact_dirs,
    write_csv,
    write_json,
)
from utils.course import chapter_by_no

ART = ensure_chapter_artifact_dirs(CHAPTER_NO, BOOK_ROOT)
FIG = ART["figures"]
HTML = ART["html"]
CHECK = ART["checks"]
TABLE = ART["tables"]
DATA = ART["data"]

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 160,
    "svg.fonttype": "none",
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
})

ARTIFACTS_WRITTEN: list[dict[str, str]] = []
CHECK_FILES: list[Path] = []

# These were produced by the legacy generic scaffold. They are intentionally
# removed on rerun so chapter-21 only advertises concept-named artifacts.
for stale in [FIG / "concept_configuration.svg", FIG / "parameter_experiment.svg"]:
    if stale.exists():
        stale.unlink()

def remember(path: Path, description: str, kind: str) -> Path:
    rel = book_relative(path, BOOK_ROOT)
    row = {"path": rel, "kind": kind, "description": description}
    for old in ARTIFACTS_WRITTEN:
        if old["path"] == rel:
            old.update(row)
            return path
    ARTIFACTS_WRITTEN.append(row)
    return path

def write_check(filename: str, payload: dict) -> Path:
    path = write_json(CHECK / filename, payload)
    CHECK_FILES.append(path)
    return remember(path, payload.get("concept", filename), "checks")

def save_figure(fig, filename: str, description: str) -> Path:
    path = FIG / filename
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return remember(path, description, "figures")

def write_table(filename: str, rows: list[dict], description: str) -> Path:
    path = write_csv(TABLE / filename, rows)
    return remember(path, description, "tables")

def heawood_real(chi: int | float) -> float:
    return (7 + math.sqrt(49 - 24 * chi)) / 2

def heawood_bound(chi: int | float) -> int:
    return math.floor(heawood_real(chi))

def orientable_chi(genus: int) -> int:
    return 2 - 2 * genus

def nonorientable_chi(crosscaps: int) -> int:
    return 2 - crosscaps

def exact_chromatic_number(G: nx.Graph) -> int:
    nodes = sorted(G.nodes(), key=lambda n: G.degree[n], reverse=True)
    colors: dict = {}
    def can_color_with(k: int, i: int = 0) -> bool:
        if i == len(nodes):
            return True
        node = nodes[i]
        forbidden = {colors[nbr] for nbr in G.neighbors(node) if nbr in colors}
        for color in range(k):
            if color not in forbidden:
                colors[node] = color
                if can_color_with(k, i + 1):
                    return True
                del colors[node]
        return False
    for k in range(1, len(nodes) + 1):
        colors.clear()
        if can_color_with(k):
            return k
    raise RuntimeError("unreachable")

def clique_number(G: nx.Graph) -> int:
    return max((len(c) for c in nx.find_cliques(G)), default=0)

def orientable_genus_complete_graph(n: int) -> int:
    if n <= 4:
        return 0
    return math.ceil((n - 3) * (n - 4) / 12)

def nonorientable_genus_complete_graph(n: int) -> int:
    if n <= 4:
        return 0
    if n == 7:
        return 3  # the Klein bottle is the classical exception
    return math.ceil((n - 3) * (n - 4) / 6)

source_span = {
    "concept": "source span inspected",
    "chapter": CHAPTER_NO,
    "printed_pages": "379-395",
    "pdf_pages": "397-413",
    "source_map_note": "AGENTS.md gives pdf_page = printed_page + 18; rendered PDF pages were inspected privately from the scanned source.",
    "topics": [
        "orientable surfaces",
        "nonorientable surfaces",
        "regular maps",
        "four-color problem",
        "six-color theorem",
        "sufficient colors for arbitrary surfaces",
        "surfaces requiring full color counts",
    ],
}
write_check("source-span.json", source_span)
chapter = chapter_by_no(CHAPTER_NO)
display(Markdown(f"Loaded **{chapter['title']}**. Artifacts will be written under `{book_relative(ART['figures'].parent, BOOK_ROOT)}`."))


Loaded **Topology of Surfaces**. Artifacts will be written under `artifacts/chapter-21`.

## 1. Orientable and Nonorientable Surfaces

The chapter starts by separating surface classification into two moves. Adding a handle preserves orientability and lowers `chi` by 2. Adding a cross-cap creates a one-sided feature and lowers `chi` by 1. The table and diagram below do not try to model every surface geometrically; they make the bookkeeping visible, because the coloring theorems later depend only on `chi` and on a small number of exceptional surfaces.


In [2]:
surface_rows = [
    {"surface": "sphere", "family": "orientable", "parameter": "g=0", "chi": orientable_chi(0), "heawood_formula": heawood_bound(2), "known_chromatic_bound": 4, "note": "four-color theorem is sharp via tetrahedron faces"},
    {"surface": "torus", "family": "orientable", "parameter": "g=1", "chi": orientable_chi(1), "heawood_formula": heawood_bound(0), "known_chromatic_bound": 7, "note": "Heawood seven-region torus map is sharp"},
    {"surface": "double torus", "family": "orientable", "parameter": "g=2", "chi": orientable_chi(2), "heawood_formula": heawood_bound(-2), "known_chromatic_bound": 8, "note": "complete graph K8 is the first genus-two witness"},
    {"surface": "projective plane", "family": "nonorientable", "parameter": "q=1", "chi": nonorientable_chi(1), "heawood_formula": heawood_bound(1), "known_chromatic_bound": 6, "note": "six colors are necessary and sufficient"},
    {"surface": "Klein bottle", "family": "nonorientable", "parameter": "q=2", "chi": nonorientable_chi(2), "heawood_formula": heawood_bound(0), "known_chromatic_bound": 6, "note": "exception: formula gives 7 but six suffice"},
    {"surface": "three cross-caps", "family": "nonorientable", "parameter": "q=3", "chi": nonorientable_chi(3), "heawood_formula": heawood_bound(-1), "known_chromatic_bound": 7, "note": "K7 becomes a sharp nonorientable witness"},
]
surface_table = write_table("surface-classification.csv", surface_rows, "orientable and nonorientable surface classification data")

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.1))
ax = axes[0]
g_vals = np.arange(0, 5)
q_vals = np.arange(1, 8)
ax.plot(g_vals, [orientable_chi(g) for g in g_vals], marker="o", label="orientable: chi = 2 - 2g", color="#245a7c")
ax.plot(q_vals, [nonorientable_chi(q) for q in q_vals], marker="s", label="nonorientable: chi = 2 - q", color="#a23b3b")
ax.axhline(0, color="0.7", lw=0.8)
ax.set_xlabel("handle genus g or cross-caps q")
ax.set_ylabel("Euler characteristic chi")
ax.set_title("Two ways chi changes")
ax.legend(frameon=False, loc="upper right", fontsize=8)
ax.grid(True, alpha=0.22)

ax = axes[1]
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Cell moves behind the formulas")
circle = patches.Circle((0.2, 1.1), 0.36, fill=False, lw=1.6, color="#333333")
ax.add_patch(circle)
ax.text(0.2, 1.1, "sphere\nchi=2", ha="center", va="center")
ax.add_patch(patches.FancyArrowPatch((0.58, 1.1), (1.34, 1.1), arrowstyle="-|>", mutation_scale=12, lw=1.2))
ax.add_patch(patches.Ellipse((1.75, 1.1), 0.8, 0.46, fill=False, lw=1.5, color="#245a7c"))
ax.add_patch(patches.Ellipse((1.75, 1.1), 0.36, 0.18, fill=False, lw=1.3, color="#245a7c"))
ax.text(1.75, 1.72, "add handle", ha="center", color="#245a7c")
ax.text(1.75, 0.45, "chi -> chi - 2", ha="center", color="#245a7c")
ax.add_patch(patches.FancyArrowPatch((0.2, 0.72), (0.2, -0.05), arrowstyle="-|>", mutation_scale=12, lw=1.2))
ax.add_patch(patches.Rectangle((-0.12, -0.72), 0.64, 0.42, fill=False, lw=1.5, color="#a23b3b"))
ax.add_patch(patches.FancyArrowPatch((-0.06, -0.3), (0.46, -0.3), arrowstyle="->", mutation_scale=10, color="#a23b3b"))
ax.add_patch(patches.FancyArrowPatch((-0.06, -0.72), (0.46, -0.72), arrowstyle="->", mutation_scale=10, color="#a23b3b"))
ax.text(0.2, -0.1, "add cross-cap", ha="center", color="#a23b3b")
ax.text(0.2, -0.94, "chi -> chi - 1", ha="center", color="#a23b3b")
ax.set_xlim(-0.35, 2.35)
ax.set_ylim(-1.1, 1.9)

ax = axes[2]
ax.axis("off")
ax.set_title("Sample surfaces used later")
cell_text = [[row["surface"], row["parameter"], row["chi"], row["known_chromatic_bound"]] for row in surface_rows]
table = ax.table(cellText=cell_text, colLabels=["surface", "param", "chi", "colors"], loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.35)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor("#e8eef2")
    elif c == 2 and cell.get_text().get_text() == "0":
        cell.set_facecolor("#fff4cc")

surface_fig = save_figure(fig, "surface-characteristic-dictionary.svg", "surface classification, Euler characteristic, and color-count dictionary")
surface_checks = {
    "concept": "Euler characteristic formulas for orientable and nonorientable closed surfaces",
    "orientable_values": {f"g={g}": orientable_chi(g) for g in range(5)},
    "nonorientable_values": {f"q={q}": nonorientable_chi(q) for q in range(1, 8)},
    "handle_delta": orientable_chi(2) - orientable_chi(1),
    "crosscap_delta": nonorientable_chi(3) - nonorientable_chi(2),
    "stale_artifacts_removed": [not (FIG / name).exists() for name in ["concept_configuration.svg", "parameter_experiment.svg"]],
}
assert surface_checks["handle_delta"] == -2
assert surface_checks["crosscap_delta"] == -1
assert all(surface_checks["stale_artifacts_removed"])
write_check("surface-euler-characteristics.json", surface_checks)

display(pd.DataFrame(surface_rows))
display_artifact(surface_fig, width=900)


,surface,family,parameter,chi,heawood_formula,known_chromatic_bound,note
0,sphere,orientable,g=0,2,4,4,four-color theorem is sharp via tetrahedron faces
1,torus,orientable,g=1,0,7,7,Heawood seven-region torus map is sharp
2,double torus,orientable,g=2,-2,8,8,complete graph K8 is the first genus-two witness
3,projective plane,nonorientable,q=1,1,6,6,six colors are necessary and sufficient
4,Klein bottle,nonorientable,q=2,0,7,6,exception: formula gives 7 but six suffice
5,three cross-caps,nonorientable,q=3,-1,7,7,K7 becomes a sharp nonorientable witness


### Orientability as a Transport Test

A local arrow carried once around an orientable loop returns compatible with itself. On a Mobius band it returns reversed. The Plotly artifact makes that test spatial: rotate the scene and compare the arrows along the middle loop. The torus arrows close with dot product `+1`; the Mobius cross-strip arrows close with dot product `-1`.


In [3]:
u = np.linspace(0, 2 * np.pi, 96)
v = np.linspace(0, 2 * np.pi, 36)
U, V = np.meshgrid(u, v)
R, r = 1.6, 0.42
Xt = (R + r * np.cos(V)) * np.cos(U)
Yt = (R + r * np.cos(V)) * np.sin(U)
Zt = r * np.sin(V)

s = np.linspace(-0.34, 0.34, 24)
Um, S = np.meshgrid(u, s)
Xm = (R + S * np.cos(Um / 2)) * np.cos(Um)
Ym = (R + S * np.cos(Um / 2)) * np.sin(Um)
Zm = S * np.sin(Um / 2)

sample_u = np.linspace(0, 2 * np.pi, 9)
torus_centers = np.column_stack([R * np.cos(sample_u), R * np.sin(sample_u), np.zeros_like(sample_u)])
torus_dirs = np.column_stack([np.cos(sample_u), np.sin(sample_u), np.zeros_like(sample_u)])
mobius_centers = torus_centers.copy()
mobius_dirs = np.column_stack([
    np.cos(sample_u / 2) * np.cos(sample_u),
    np.cos(sample_u / 2) * np.sin(sample_u),
    np.sin(sample_u / 2),
])

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]], subplot_titles=("Torus: frame closes", "Mobius band: frame reverses"))
fig.add_trace(go.Surface(x=Xt, y=Yt, z=Zt, opacity=0.55, colorscale="Blues", showscale=False, name="torus"), row=1, col=1)
fig.add_trace(go.Scatter3d(x=torus_centers[:, 0], y=torus_centers[:, 1], z=torus_centers[:, 2], mode="lines", line=dict(color="#1b3a57", width=6), showlegend=False), row=1, col=1)
fig.add_trace(go.Cone(x=torus_centers[:, 0], y=torus_centers[:, 1], z=torus_centers[:, 2], u=0.35*torus_dirs[:, 0], v=0.35*torus_dirs[:, 1], w=0.35*torus_dirs[:, 2], sizemode="absolute", sizeref=0.23, anchor="tail", colorscale=[[0, "#1b3a57"], [1, "#1b3a57"]], showscale=False), row=1, col=1)
fig.add_trace(go.Surface(x=Xm, y=Ym, z=Zm, opacity=0.6, colorscale="Reds", showscale=False, name="mobius"), row=1, col=2)
fig.add_trace(go.Scatter3d(x=mobius_centers[:, 0], y=mobius_centers[:, 1], z=mobius_centers[:, 2], mode="lines", line=dict(color="#7a1f1f", width=6), showlegend=False), row=1, col=2)
fig.add_trace(go.Cone(x=mobius_centers[:, 0], y=mobius_centers[:, 1], z=mobius_centers[:, 2], u=0.35*mobius_dirs[:, 0], v=0.35*mobius_dirs[:, 1], w=0.35*mobius_dirs[:, 2], sizemode="absolute", sizeref=0.23, anchor="tail", colorscale=[[0, "#7a1f1f"], [1, "#7a1f1f"]], showscale=False), row=1, col=2)
fig.update_layout(height=560, margin=dict(l=0, r=0, t=48, b=0), scene=dict(aspectmode="data"), scene2=dict(aspectmode="data"))
for scene_name in ["scene", "scene2"]:
    fig.layout[scene_name].xaxis.visible = False
    fig.layout[scene_name].yaxis.visible = False
    fig.layout[scene_name].zaxis.visible = False

orient_html = HTML / "orientability-frame-transport.html"
fig.write_html(orient_html, include_plotlyjs=True, full_html=True)
remember(orient_html, "interactive frame-transport test for orientability", "html")

torus_dot = float(np.dot(torus_dirs[0], torus_dirs[-1]))
mobius_dot = float(np.dot(mobius_dirs[0], mobius_dirs[-1]))
orient_checks = {
    "concept": "orientability via transported local frame",
    "torus_start_end_dot": torus_dot,
    "mobius_start_end_dot": mobius_dot,
    "interpretation": "positive closure for torus, negative closure for Mobius band cross-direction",
}
assert abs(torus_dot - 1.0) < 1e-12
assert abs(mobius_dot + 1.0) < 1e-12
write_check("orientability-frame-checks.json", orient_checks)

display_artifact(orient_html, width="100%")
orient_checks


{'concept': 'orientability via transported local frame',
 'torus_start_end_dot': 1.0,
 'mobius_start_end_dot': -1.0,
 'interpretation': 'positive closure for torus, negative closure for Mobius band cross-direction'}

## 2. Regular Maps and the `{p, q}` Count

A regular map keeps the same local incidence pattern everywhere: `p` edges bound each face and `q` edges meet at each vertex. Counting incidences gives `pF = 2E = qV`. Combining those equations with Euler's formula recovers the chapter's regular-map count formula. The special case `chi = 0` is exactly where the denominator can vanish, which is why the torus has infinite regular-map families of types `{4,4}`, `{3,6}`, and `{6,3}`.


In [4]:
p, q, r_sym, chi_sym = sp.symbols("p q r chi")
V_expr = 2 * p * r_sym
E_expr = p * q * r_sym
F_expr = 2 * q * r_sym
euler_expr = sp.expand(V_expr - E_expr + F_expr)
r_solution = sp.solve(sp.Eq(euler_expr, chi_sym), r_sym)[0]
assert sp.simplify(r_solution - chi_sym / (2*p + 2*q - p*q)) == 0
assert sp.simplify(p*F_expr - 2*E_expr) == 0
assert sp.simplify(q*V_expr - 2*E_expr) == 0

spherical_types = [(3, 3, "tetrahedron"), (4, 3, "cube"), (3, 4, "octahedron"), (5, 3, "dodecahedron"), (3, 5, "icosahedron")]
regular_rows = []
for pp, qq, name in spherical_types:
    rr = sp.Rational(2, 2*pp + 2*qq - pp*qq)
    regular_rows.append({
        "type": f"{{{pp},{qq}}}", "surface": "sphere", "name": name, "chi": 2, "r": str(rr),
        "V": int(2*pp*rr), "E": int(pp*qq*rr), "F": int(2*qq*rr),
    })
for pp, qq, b, c, family in [(4, 4, 3, 2, "square torus"), (3, 6, 2, 1, "triangular torus"), (6, 3, 2, 1, "hexagonal torus dual")]:
    if (pp, qq) == (4, 4):
        n = b*b + c*c
        Vv, Ee, Ff = n, 2*n, n
    elif (pp, qq) == (3, 6):
        n = b*b + b*c + c*c
        Vv, Ee, Ff = n, 3*n, 2*n
    else:
        n = b*b + b*c + c*c
        Vv, Ee, Ff = 2*n, 3*n, n
    regular_rows.append({"type": f"{{{pp},{qq}}}_{{{b},{c}}}", "surface": "torus", "name": family, "chi": 0, "r": "free", "V": Vv, "E": Ee, "F": Ff})
regular_table = write_table("regular-map-counts.csv", regular_rows, "regular map incidence counts and torus families")
regular_checks = {
    "concept": "regular map incidence equations",
    "formula": {"V": "2pr", "E": "pqr", "F": "2qr", "r": "chi/(2p+2q-pq)"},
    "symbolic_euler_expression": str(euler_expr),
    "r_solution": str(r_solution),
    "rows_checked": len(regular_rows),
    "all_rows_satisfy_euler": all(row["V"] - row["E"] + row["F"] == row["chi"] for row in regular_rows),
    "torus_zero_denominator_types": ["{4,4}", "{3,6}", "{6,3}"],
}
assert regular_checks["all_rows_satisfy_euler"]
write_check("regular-map-formula-checks.json", regular_checks)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
ax = axes[0]
for row in regular_rows[:5]:
    pp, qq = map(int, row["type"].strip("{}").split(","))
    ax.scatter(pp, qq, s=70, color="#376f8c")
    ax.text(pp + 0.05, qq + 0.05, row["name"], fontsize=8)
ax.plot([2.4, 5.4], [6, 2.4], color="0.78", lw=1)
ax.set_xlim(2.5, 5.6)
ax.set_ylim(2.5, 5.6)
ax.set_xlabel("p: edges around a face")
ax.set_ylabel("q: edges at a vertex")
ax.set_title("Spherical regular map types")
ax.grid(True, alpha=0.25)

ax = axes[1]
ax.set_aspect("equal")
ax.set_title("Square torus family {4,4}_{b,c}")
for i in range(5):
    ax.plot([0, 4], [i, i], color="0.82", lw=0.8)
    ax.plot([i, i], [0, 4], color="0.82", lw=0.8)
ax.add_patch(patches.Polygon([[0,0],[3,2],[1,5],[-2,3]], closed=True, fill=False, lw=2.0, color="#245a7c"))
ax.arrow(0, 0, 3, 2, head_width=0.12, length_includes_head=True, color="#245a7c")
ax.arrow(0, 0, -2, 3, head_width=0.12, length_includes_head=True, color="#245a7c")
ax.text(1.45, 0.82, "(b,c)=(3,2)", color="#245a7c", fontsize=8)
ax.text(0.05, 4.55, "V=b^2+c^2=13, E=26, F=13", fontsize=8)
ax.set_xlim(-2.4, 4.4)
ax.set_ylim(-0.3, 5.4)
ax.axis("off")

ax = axes[2]
ax.set_aspect("equal")
ax.set_title("Triangular torus family {3,6}_{b,c}")
for i in range(-2, 6):
    ax.plot([i, i+4], [0, 0], color="0.82", lw=0.8)
    ax.plot([i, i+2], [0, 3.46], color="0.82", lw=0.8)
    ax.plot([i, i-2], [0, 3.46], color="0.82", lw=0.8)
tri_a = np.array([2, 1])
tri_b = np.array([1, 2])
B = np.array([[1, 0.5], [0, np.sqrt(3)/2]])
poly = np.vstack([B @ np.array([0, 0]), B @ tri_a, B @ (tri_a + tri_b), B @ tri_b])
ax.add_patch(patches.Polygon(poly, closed=True, fill=False, lw=2.0, color="#7f3b2e"))
ax.text(0.1, 3.6, "V=b^2+bc+c^2=7\nE=21, F=14", fontsize=8, color="#7f3b2e")
ax.set_xlim(-1.0, 4.2)
ax.set_ylim(-0.4, 4.2)
ax.axis("off")
regular_fig = save_figure(fig, "regular-map-counts-and-duals.svg", "regular map formula, duality, and torus families")

display(pd.DataFrame(regular_rows))
display_artifact(regular_fig, width=950)


,type,surface,name,chi,r,V,E,F
0,"{3,3}",sphere,tetrahedron,2,2/3,4,6,4
1,"{4,3}",sphere,cube,2,1,8,12,6
2,"{3,4}",sphere,octahedron,2,1,6,12,8
3,"{5,3}",sphere,dodecahedron,2,2,20,30,12
4,"{3,5}",sphere,icosahedron,2,2,12,30,20
5,"{4,4}_{3,2}",torus,square torus,0,free,13,26,13
6,"{3,6}_{2,1}",torus,triangular torus,0,free,7,21,14
7,"{6,3}_{2,1}",torus,hexagonal torus dual,0,free,14,21,7


## 3. The Four-Color Problem as Face-Adjacency Coloring

For coloring maps, the useful graph is not the embedded edge graph itself but the face-adjacency graph: one node for each face, with an edge when two faces share a boundary arc. The four-color problem asks whether every sphere or plane map has a face coloring with at most four colors. The source treats this as a problem; modern mathematics has a computer-assisted proof of the four-color theorem. The notebook will not reprove that theorem. Instead, it makes the graph translation and sharp small examples explicit before proving the weaker six-color theorem.


In [5]:
K4 = nx.complete_graph(4)
W6 = nx.wheel_graph(6)
K5 = nx.complete_graph(5)
four_color_graphs = [("tetrahedron face graph K4", K4), ("five-neighbor wheel", W6), ("K5 obstruction", K5)]
chromatic_rows = []
for name, G in four_color_graphs:
    chromatic_rows.append({
        "graph": name,
        "vertices": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "planar": nx.check_planarity(G)[0],
        "chromatic_number": exact_chromatic_number(G),
        "clique_number": clique_number(G),
    })

palette = ["#4676a3", "#d98f35", "#6aa66a", "#b65050", "#8a6fb1", "#b7a33a"]
fig, axes = plt.subplots(1, 3, figsize=(12.8, 3.8))
for ax, (name, G), row in zip(axes, four_color_graphs, chromatic_rows):
    pos = nx.circular_layout(G)
    coloring = nx.coloring.greedy_color(G, strategy="largest_first")
    node_colors = [palette[coloring[n] % len(palette)] for n in G.nodes()]
    nx.draw_networkx_edges(G, pos, ax=ax, width=1.2, edge_color="0.45")
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=520, edgecolors="white", linewidths=1.2)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_color="white")
    ax.set_title(f"{name}\nchi_color={row['chromatic_number']}, planar={row['planar']}")
    ax.axis("off")
four_fig = save_figure(fig, "four-color-face-adjacency-examples.svg", "face-adjacency graph coloring examples for the four-color problem")

four_checks = {
    "concept": "map coloring as face-adjacency graph coloring",
    "examples": chromatic_rows,
    "interpretation": "K4 shows four colors can be necessary on the sphere; K5 shows why clique size matters but is not planar.",
}
assert chromatic_rows[0]["chromatic_number"] == 4
assert chromatic_rows[0]["planar"]
assert not chromatic_rows[2]["planar"]
write_check("four-color-example-checks.json", four_checks)

display(pd.DataFrame(chromatic_rows))
display_artifact(four_fig, width=900)


,graph,vertices,edges,planar,chromatic_number,clique_number
0,tetrahedron face graph K4,4,6,True,4,4
1,five-neighbor wheel,6,10,True,4,3
2,K5 obstruction,5,10,False,5,5


## 4. The Six-Color Theorem

The six-color theorem uses only the part of Euler's formula that is easy to check computationally. If every vertex has degree at least three, then `2E >= 3V`. Combining this with `chi = V - E + F` gives `2E/F <= 6(1 - chi/F)`. On the sphere and projective plane, `chi > 0`, so the average number of sides per face is less than 6; at least one face has at most five neighbors. Remove that face, color the rest by induction, then put it back using a sixth color if necessary.


In [6]:
angles = np.linspace(np.pi/2, np.pi/2 + 2*np.pi, 6)[:-1]
outer = np.column_stack([np.cos(angles), np.sin(angles)])
inner = 0.42 * outer
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
for ax, title, show_center in [(axes[0], "before removal", True), (axes[1], "after shrinking the face", False)]:
    ax.set_aspect("equal")
    ax.axis("off")
    for i in range(5):
        poly = np.vstack([[0, 0], outer[i], outer[(i+1) % 5]]) if show_center else np.vstack([inner[i], outer[i], outer[(i+1)%5], inner[(i+1)%5]])
        ax.add_patch(patches.Polygon(poly, closed=True, facecolor=palette[i], edgecolor="white", alpha=0.85, lw=1.2))
        label_xy = 0.72 * (outer[i] + outer[(i+1)%5]) / 2
        ax.text(label_xy[0], label_xy[1], str(i+1), ha="center", va="center", color="white", weight="bold")
    if show_center:
        ax.add_patch(patches.Polygon(inner, closed=True, facecolor="#f3f3f0", edgecolor="0.2", lw=1.4, hatch="////"))
        ax.text(0, 0, "face\nF", ha="center", va="center")
    else:
        ax.plot(inner[:,0], inner[:,1], color="0.2", lw=1.4)
        ax.text(0, 0, "point", ha="center", va="center")
    ax.set_title(title)
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.15, 1.25)
axes[0].text(0, -1.03, "five neighbors may use colors 1-5", ha="center", fontsize=9)
axes[1].text(0, -1.03, "induction colors the smaller map", ha="center", fontsize=9)
six_fig = save_figure(fig, "six-color-induction-step.svg", "six-color theorem induction step with a low-edge face")

F_sym, chi_pos = sp.symbols("F chi", positive=True)
E_bound = 3 * (F_sym - chi_pos)
avg_bound = sp.simplify(2 * E_bound / F_sym)
N_sym = sp.symbols("N", positive=True)
chi_from_N = sp.simplify((7*N_sym - N_sym**2) / 6)
heawood_identity = sp.simplify(6 * (1 - chi_from_N / N_sym) - (N_sym - 1))
assert heawood_identity == 0

six_rows = []
for chi_val, label in [(2, "sphere"), (1, "projective plane")]:
    for F_val in [6, 12, 20]:
        avg_max = float(6 * (1 - chi_val / F_val))
        six_rows.append({"surface": label, "chi": chi_val, "F": F_val, "average_face_edges_upper_bound": avg_max, "forces_face_with_at_most": math.floor(avg_max)})
        assert avg_max < 6
six_checks = {
    "concept": "six-color theorem low-edge-face induction",
    "degree_inequality": "2E >= 3V",
    "average_face_edge_bound": str(avg_bound),
    "positive_chi_samples": six_rows,
    "heawood_quadratic_identity_for_extension": str(heawood_identity),
}
write_check("six-color-theorem-checks.json", six_checks)

display(pd.DataFrame(six_rows))
display_artifact(six_fig, width=760)


,surface,chi,F,average_face_edges_upper_bound,forces_face_with_at_most
0,sphere,2,6,4.0,4
1,sphere,2,12,5.0,5
2,sphere,2,20,5.4,5
3,projective plane,1,6,5.0,5
4,projective plane,1,12,5.5,5
5,projective plane,1,20,5.7,5


## 5. A Sufficient Number of Colors for Any Surface

For a surface with characteristic `chi < 2`, the source uses Heawood's expression

`N = (7 + sqrt(49 - 24 chi)) / 2`

and proves that the integer part is sufficient. The proof is the six-color induction with a larger budget: the quadratic equation `N^2 - 7N + 6chi = 0` is exactly what makes the average-face inequality produce a removable face with at most `[N]-1` neighbors.


In [7]:
heawood_rows = []
for chi_val in range(2, -10, -1):
    real = heawood_real(chi_val)
    floor_val = heawood_bound(chi_val)
    heawood_rows.append({
        "chi": chi_val,
        "N_real": round(real, 6),
        "floor_N": floor_val,
        "surface_hint": {2: "sphere", 1: "projective plane", 0: "torus or Klein bottle", -1: "three cross-caps"}.get(chi_val, "more complicated surface"),
        "quadratic_residual": round(real*real - 7*real + 6*chi_val, 10),
    })
heawood_table = write_table("heawood-bound-table.csv", heawood_rows, "Heawood sufficient color counts by Euler characteristic")

fig, ax = plt.subplots(figsize=(9.5, 4.5))
chis = [row["chi"] for row in heawood_rows]
floors = [row["floor_N"] for row in heawood_rows]
reals = [row["N_real"] for row in heawood_rows]
ax.plot(chis, reals, color="#4b6f91", lw=2.0, label="real Heawood expression")
ax.step(chis, floors, where="mid", color="#b04c3b", lw=2.0, label="integer part used for coloring")
ax.scatter(chis, floors, color="#b04c3b", zorder=3)
for row in heawood_rows[:5]:
    ax.annotate(row["surface_hint"], (row["chi"], row["floor_N"]), textcoords="offset points", xytext=(4, 7), fontsize=8)
ax.set_xlabel("Euler characteristic chi")
ax.set_ylabel("sufficient colors")
ax.set_title("Heawood bound as chi decreases")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
heawood_fig = save_figure(fig, "heawood-bound-by-characteristic.svg", "Heawood sufficient color bound by Euler characteristic")

heawood_checks = {
    "concept": "Heawood sufficient color bound",
    "identity": "N^2 - 7N + 6chi = 0",
    "table_residual_max_abs": max(abs(row["quadratic_residual"]) for row in heawood_rows),
    "values": heawood_rows,
}
assert heawood_checks["table_residual_max_abs"] < 1e-7
write_check("heawood-bound-checks.json", heawood_checks)

display(pd.DataFrame(heawood_rows))
display_artifact(heawood_fig, width=820)


,chi,N_real,floor_N,surface_hint,quadratic_residual
0,2,4.000000,4,sphere,0.0
1,1,6.000000,6,projective plane,0.0
2,0,7.000000,7,torus or Klein bottle,0.0
3,-1,7.772002,7,three cross-caps,-0.0
4,-2,8.424429,8,more complicated surface,0.0
5,-3,9.000000,9,more complicated surface,0.0
6,-4,9.520797,9,more complicated surface,0.0
7,-5,10.000000,10,more complicated surface,0.0
8,-6,10.446222,10,more complicated surface,-0.0
9,-7,10.865460,10,more complicated surface,0.0


## 6. Surfaces That Need the Full Number of Colors

A full-color witness is a map whose face-adjacency graph contains a complete graph of the required size. Complete graphs are useful because their chromatic number is visible: `K_n` needs `n` colors. The subtlety is not the coloring of `K_n`; it is whether such a graph can be embedded on the surface as a map. The table below records the classical witnesses used by the chapter's endgame, including the Klein-bottle exception where Heawood's formula predicts seven but the sharp count is six.


In [8]:
witness_rows = [
    {"surface": "sphere", "chi": 2, "formula_bound": 4, "sharp_count": 4, "witness_graph": "K4", "orientable_genus_needed": orientable_genus_complete_graph(4), "nonorientable_crosscaps_needed": nonorientable_genus_complete_graph(4), "note": "tetrahedron face adjacency"},
    {"surface": "projective plane", "chi": 1, "formula_bound": 6, "sharp_count": 6, "witness_graph": "K6", "orientable_genus_needed": orientable_genus_complete_graph(6), "nonorientable_crosscaps_needed": nonorientable_genus_complete_graph(6), "note": "six colors necessary"},
    {"surface": "torus", "chi": 0, "formula_bound": 7, "sharp_count": 7, "witness_graph": "K7", "orientable_genus_needed": orientable_genus_complete_graph(7), "nonorientable_crosscaps_needed": nonorientable_genus_complete_graph(7), "note": "Heawood seven-region map"},
    {"surface": "Klein bottle", "chi": 0, "formula_bound": 7, "sharp_count": 6, "witness_graph": "exception", "orientable_genus_needed": "n/a", "nonorientable_crosscaps_needed": "n/a", "note": "Franklin's theorem: six suffice"},
    {"surface": "three cross-caps", "chi": -1, "formula_bound": 7, "sharp_count": 7, "witness_graph": "K7", "orientable_genus_needed": orientable_genus_complete_graph(7), "nonorientable_crosscaps_needed": nonorientable_genus_complete_graph(7), "note": "K7 avoids the Klein-bottle exception"},
    {"surface": "orientable genus 2", "chi": -2, "formula_bound": 8, "sharp_count": 8, "witness_graph": "K8", "orientable_genus_needed": orientable_genus_complete_graph(8), "nonorientable_crosscaps_needed": nonorientable_genus_complete_graph(8), "note": "K8 starts at genus two"},
]
witness_table = write_table("full-color-witnesses.csv", witness_rows, "complete-graph witnesses for full surface color counts")

fig, axes = plt.subplots(2, 3, figsize=(12.5, 7.1))
for ax, row in zip(axes.flat, witness_rows):
    ax.axis("off")
    ax.set_title(f"{row['surface']}\nsharp={row['sharp_count']}, formula={row['formula_bound']}", fontsize=10)
    if row["witness_graph"].startswith("K"):
        n = int(row["witness_graph"][1:])
        G = nx.complete_graph(n)
        pos = nx.circular_layout(G)
        colors = [palette[i % len(palette)] for i in range(n)]
        nx.draw_networkx_edges(G, pos, ax=ax, width=0.9, edge_color="0.55", alpha=0.75)
        nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=290, edgecolors="white", linewidths=0.8)
        nx.draw_networkx_labels(G, pos, ax=ax, font_size=7, font_color="white")
        ax.text(0, -1.28, f"{row['witness_graph']} needs {exact_chromatic_number(G)} colors", ha="center", fontsize=8)
    else:
        ax.text(0.5, 0.58, "Klein bottle\nexception", transform=ax.transAxes, ha="center", va="center", fontsize=13, color="#7a1f1f")
        ax.text(0.5, 0.32, "formula gives 7;\nsharp count is 6", transform=ax.transAxes, ha="center", va="center", fontsize=9)
full_fig = save_figure(fig, "full-color-witness-cliques.svg", "complete graph witnesses and Klein-bottle exception for full color counts")

witness_checks = {
    "concept": "full color count witnesses by complete graphs",
    "rows": witness_rows,
    "chromatic_numbers": {f"K{n}": exact_chromatic_number(nx.complete_graph(n)) for n in [4, 6, 7, 8]},
    "genus_formulas": {
        "orientable_Kn": "ceil((n-3)(n-4)/12)",
        "nonorientable_Kn": "ceil((n-3)(n-4)/6), except K7 needs three cross-caps",
    },
}
assert witness_checks["chromatic_numbers"] == {"K4": 4, "K6": 6, "K7": 7, "K8": 8}
assert orientable_genus_complete_graph(7) == 1
assert orientable_genus_complete_graph(8) == 2
assert nonorientable_genus_complete_graph(7) == 3
write_check("full-color-witness-checks.json", witness_checks)

display(pd.DataFrame(witness_rows))
display_artifact(full_fig, width=920)


,surface,chi,formula_bound,sharp_count,witness_graph,orientable_genus_needed,nonorientable_crosscaps_needed,note
0,sphere,2,4,4,K4,0,0,tetrahedron face adjacency
1,projective plane,1,6,6,K6,1,1,six colors necessary
2,torus,0,7,7,K7,1,3,Heawood seven-region map
3,Klein bottle,0,7,6,exception,n/a,n/a,Franklin's theorem: six suffice
4,three cross-caps,-1,7,7,K7,1,3,K7 avoids the Klein-bottle exception
5,orientable genus 2,-2,8,8,K8,2,4,K8 starts at genus two


## Applied Lab: Which Complete Graph Fits?

The exploration below varies the surface and asks whether the complete graph matching Heawood's bound has a known genus no larger than the surface's genus. This is not a replacement for an embedding construction. It is a sanity filter: if the required complete graph needs more genus than the surface has, that graph cannot be the witness on that surface.


In [9]:
lab_rows = []
for g in range(0, 7):
    chi_val = orientable_chi(g)
    H = heawood_bound(chi_val)
    lab_rows.append({
        "family": "orientable",
        "parameter": f"g={g}",
        "chi": chi_val,
        "heawood_floor": H,
        "candidate_graph": f"K{H}",
        "candidate_genus_needed": orientable_genus_complete_graph(H),
        "fits_by_genus_formula": orientable_genus_complete_graph(H) <= g,
        "exception_note": "none",
    })
for q_cross in range(1, 9):
    chi_val = nonorientable_chi(q_cross)
    H = heawood_bound(chi_val)
    needed = nonorientable_genus_complete_graph(H)
    lab_rows.append({
        "family": "nonorientable",
        "parameter": f"q={q_cross}",
        "chi": chi_val,
        "heawood_floor": H,
        "candidate_graph": f"K{H}",
        "candidate_genus_needed": needed,
        "fits_by_genus_formula": needed <= q_cross,
        "exception_note": "Klein bottle exception" if q_cross == 2 and H == 7 else "none",
    })
lab_table = write_table("surface-color-witness-lab.csv", lab_rows, "surface color witness genus exploration")
lab_checks = {
    "concept": "complete graph witness exploration",
    "orientable_rows_fit": all(row["fits_by_genus_formula"] for row in lab_rows if row["family"] == "orientable"),
    "klein_bottle_flagged": any(row["exception_note"] == "Klein bottle exception" for row in lab_rows),
    "rows": lab_rows,
}
assert lab_checks["orientable_rows_fit"]
assert lab_checks["klein_bottle_flagged"]
write_check("surface-color-witness-lab-checks.json", lab_checks)

display(pd.DataFrame(lab_rows))
display_artifact(lab_table)


,family,parameter,chi,heawood_floor,candidate_graph,candidate_genus_needed,fits_by_genus_formula,exception_note
0,orientable,g=0,2,4,K4,0,True,none
1,orientable,g=1,0,7,K7,1,True,none
2,orientable,g=2,-2,8,K8,2,True,none
3,orientable,g=3,-4,9,K9,3,True,none
4,orientable,g=4,-6,10,K10,4,True,none
5,orientable,g=5,-8,11,K11,5,True,none
6,orientable,g=6,-10,12,K12,6,True,none
7,nonorientable,q=1,1,6,K6,1,True,none
8,nonorientable,q=2,0,7,K7,3,False,Klein bottle exception
9,nonorientable,q=3,-1,7,K7,3,True,none


## How To Read The Coloring And Surface Checks

The coloring visuals should be read as topology first and graph theory second. Euler characteristic tells the notebook how many vertices, edges, and faces can coexist on a surface; orientability tells whether a local frame returns coherently after transport; the face-adjacency graph then records which regions are forbidden to share a color. This order prevents a common mistake: treating the Heawood number as a decorative formula rather than as a bound tied to the surface that carries the map.

The complete-graph witness table is also diagnostic. When `K_n` embeds on a surface, every pair of regions can be forced to touch, so `n` colors are genuinely necessary. The Klein-bottle exception is called out because it is exactly the kind of edge case a standalone computational course should not hide.

## Takeaways

- Euler characteristic is the chapter's shared currency: handles, cross-caps, regular maps, and coloring bounds all spend it differently.
- Orientability is not just a label. A transported local frame can return consistently or reversed, and that test distinguishes a torus from a Mobius-type one-sided surface.
- Regular maps translate into incidence equations. Away from `chi=0`, the formula determines `V`, `E`, and `F`; at `chi=0`, infinite torus families appear.
- Four colors are the sharp sphere result, but the six-color theorem is the elementary induction that follows directly from Euler's formula.
- Heawood's expression gives a sufficient color count for arbitrary surfaces, with the Klein bottle as the famous exception to the naive sharpness story.
- Full color counts are witnessed by complete face-adjacency graphs when those graphs embed on the surface.


## Final Sanity Checks

The last cell records a manifest, checks that every artifact exists and is nonempty, verifies the core identities used above, and confirms that the legacy generic figure names are absent.


In [10]:
storyboard = {
    "concept": "chapter 21 visual storyboard implemented",
    "chapter_goal": "Make topology of surfaces inspectable through Euler characteristic, orientability transport, regular-map counts, and map-coloring witnesses.",
    "source_span": source_span,
    "items": [
        "orientable and nonorientable surface dictionary",
        "interactive frame transport for orientability",
        "regular map incidence and torus family counts",
        "four-color face-adjacency examples",
        "six-color induction proof scaffold",
        "Heawood bound table and plot",
        "full-color complete-graph witnesses",
        "surface color witness applied lab",
    ],
    "libraries": {
        "Plotly": "interactive 3D surface/frame transport",
        "Matplotlib": "durable SVG proof diagrams and bound plots",
        "NetworkX": "face-adjacency graphs, coloring checks, and complete-graph witnesses",
        "SymPy": "symbolic regular-map and Heawood identities",
        "Pandas": "auditable tables displayed inline and saved as CSV",
    },
}
write_check("storyboard.json", storyboard)

assert orientable_chi(3) == -4
assert nonorientable_chi(5) == -3
assert sp.simplify((2*p*r_sym) - (p*q*r_sym) + (2*q*r_sym) - r_sym*(2*p + 2*q - p*q)) == 0
assert sp.simplify(6 * (1 - chi_from_N / N_sym) - (N_sym - 1)) == 0
assert heawood_bound(2) == 4
assert heawood_bound(1) == 6
assert heawood_bound(0) == 7
assert exact_chromatic_number(nx.complete_graph(7)) == 7
assert orientable_genus_complete_graph(7) == 1
assert nonorientable_genus_complete_graph(7) == 3
assert not (FIG / "concept_configuration.svg").exists()
assert not (FIG / "parameter_experiment.svg").exists()

artifact_paths = [BOOK_ROOT / row["path"] for row in ARTIFACTS_WRITTEN]
assert_artifacts(artifact_paths, min_bytes=100)
final_sanity_payload = {
    "concept": "chapter 21 final sanity checks",
    "artifact_count_before_summary_and_manifest": len(ARTIFACTS_WRITTEN),
    "check_count_before_summary": len(CHECK_FILES),
    "all_artifacts_nonempty": all(path.exists() and path.stat().st_size > 20 for path in artifact_paths),
    "core_checks": {
        "orientable_genus_3_chi": orientable_chi(3),
        "nonorientable_q5_chi": nonorientable_chi(5),
        "heawood_values": {"chi=2": heawood_bound(2), "chi=1": heawood_bound(1), "chi=0": heawood_bound(0), "chi=-2": heawood_bound(-2)},
        "K7_chromatic_number": exact_chromatic_number(nx.complete_graph(7)),
        "K7_orientable_genus": orientable_genus_complete_graph(7),
        "K7_nonorientable_crosscaps": nonorientable_genus_complete_graph(7),
    },
    "stale_generic_artifacts_absent": True,
}
final_sanity_path = write_check("final-sanity-summary.json", final_sanity_payload)

visual_summary_path = CHECK / "visual_summary.json"
manifest_path = TABLE / "artifact_manifest.csv"
visual_summary_row = {"path": book_relative(visual_summary_path, BOOK_ROOT), "kind": "checks", "description": "chapter 21 visual summary"}
manifest_row = {"path": book_relative(manifest_path, BOOK_ROOT), "kind": "tables", "description": "chapter 21 artifact manifest"}
predicted_manifest_rows = ARTIFACTS_WRITTEN + [visual_summary_row, manifest_row]
visual_summary = {
    "concept": "chapter 21 visual summary",
    "chapter": CHAPTER_NO,
    "title": "Topology of Surfaces",
    "source_span": {"printed_pages": "379-395", "pdf_pages": "397-413"},
    "artifact_count": len(predicted_manifest_rows),
    "checks": [book_relative(path, BOOK_ROOT) for path in CHECK_FILES] + [book_relative(visual_summary_path, BOOK_ROOT)],
    "artifacts": predicted_manifest_rows,
}
write_json(visual_summary_path, visual_summary)
CHECK_FILES.append(visual_summary_path)
remember(visual_summary_path, "chapter 21 visual summary", "checks")

manifest_rows = ARTIFACTS_WRITTEN + [manifest_row]
write_csv(manifest_path, manifest_rows)
remember(manifest_path, "chapter 21 artifact manifest", "tables")

all_paths = [BOOK_ROOT / row["path"] for row in ARTIFACTS_WRITTEN]
assert_artifacts(all_paths, min_bytes=100)
assert len({row["path"] for row in ARTIFACTS_WRITTEN}) == len(ARTIFACTS_WRITTEN)
summary = {
    "artifact_count": len(ARTIFACTS_WRITTEN),
    "check_count": len(CHECK_FILES),
    "manifest": book_relative(manifest_path, BOOK_ROOT),
    "largest_artifact_bytes": max(path.stat().st_size for path in all_paths),
    "stale_generic_artifacts_absent": True,
}
display_artifact(manifest_path)
summary


{'artifact_count': 25,
 'check_count': 12,
 'manifest': 'artifacts/chapter-21/tables/artifact_manifest.csv',
 'largest_artifact_bytes': 5095105,
 'stale_generic_artifacts_absent': True}